## Set up Graph Overlay

### Basics of Graph

In [1]:
from operator import index

import numpy as np, networkx as nx
# import watts_model as wtm, gudhi_persistence as gp, utilsA1 as utils
import os, json, pandas as pd

## Variables
base_PATH = os.path.realpath(os.path.join(os.getcwd(), "../../"))
base_PATH

'C:\\Users\\sranasin\\PycharmProjects\\NetworkModels'

In [2]:
graph = nx.read_graphml(os.path.join(base_PATH, "resources/masters33.graphml"))
graph = nx.relabel_nodes(graph, int)
schema_info = dict(json.loads(graph.graph['schema_info']))
schema_id = dict(json.loads(graph.graph['schema_id']))

for key, item in schema_id.items():
    print(f" {key}: {item} : {schema_info[key]}")

 d13: FA_std : Standard deviation of FA across streamlines
 d12: fiber_length_mean : Average fiber length (mm)
 d11: fiber_length_std : Std. deviation of fiber length (mm)
 d10: FA_mean : Mean fractional anisotropy along streamlines
 d9: number_of_fibers : Count of streamlines connecting two regions
 d7: dn_hemisphere : Left or right hemisphere
 d6: dn_name : Full region label including hemisphere
 d5: dn_fsname : FreeSurfer base region name
 d4: dn_region : Cortical or subcortical classification
 d3: dn_correspondence_id : Atlas ID used to match regions across subjects
 d2: dn_position_z : DK z-axis centroid position (in mm)
 d1: dn_position_y : DK y-axis centroid position (in mm)
 d0: dn_position_x : DK x-axis centroid position (in mm)


In [3]:
node_df = pd.DataFrame.from_dict(dict(graph.nodes(data = True)), orient = 'index')
node_df.rename(columns = {"dn_correspondence_id":"node_id", "dn_fsname": "dn_old_fsname"}, inplace = True)
# node_df = node_df.astype({'node_id':'int64'})
assert node_df['node_id'].dtype == 'int64', "Relabelling Failed"
assert node_df.index.dtype == 'int64', "Relabelling Failed"
node_df.head(5)

,dn_position_x,dn_position_y,dn_position_z,node_id,dn_region,dn_old_fsname,dn_name,dn_hemisphere
1,34.041923,82.175769,31.776923,1,cortical,lateralorbitofrontal,rh.lateralorbitofrontal,right
2,24.289760,88.730937,36.017429,2,cortical,parsorbitalis,rh.parsorbitalis,right
3,39.947195,100.095710,36.742574,3,cortical,frontalpole,rh.frontalpole,right
4,42.589852,84.868922,32.487844,4,cortical,medialorbitofrontal,rh.medialorbitofrontal,right
5,21.495468,79.767976,39.722054,5,cortical,parstriangularis,rh.parstriangularis,right


In [4]:
def edge_df_from_graph(G):
    graph_edge_df = pd.DataFrame(list(G.edges(data = True)), columns = ['source', 'target', 'attributes'])
    # graph_edge_df = graph_edge_df.astype({'source':'int64', 'target':'int64'})
    assert (graph_edge_df[['source', 'target']].dtypes == ['int64', 'int64']).all(), "Relabelling Failed"

    if graph_edge_df['attributes'].apply(lambda x: isinstance(x, dict)).all():
        attr_df = pd.json_normalize(graph_edge_df['attributes'])
        graph_edge_df = pd.concat([graph_edge_df[['source', 'target']], attr_df], axis = 1)
    return graph_edge_df

edge_df = edge_df_from_graph(graph)
edge_df.head(5)

,source,target,number_of_fibers,FA_mean,fiber_length_std,fiber_length_mean,FA_std
0,1,2,5.629108,0.178234,0.816500,15.957570,0.116143
1,1,36,17.152582,0.373044,13.548146,24.690756,0.159819
2,1,37,85.394366,0.414124,9.933109,26.245858,0.149843
3,1,7,54.953052,0.338660,6.342119,23.686359,0.179367
4,1,8,11.751174,0.511627,7.169767,30.634023,0.156943


#### Import ADNI/other Data

In [11]:
# SUVR or Volume or other Data
df = pd.read_csv(os.path.join(base_PATH, "resources/adni_pet_image_analysis/structured_files_UCBERKELEY_AMY_6MM_29Oct2025/UCBERKELEY_AMY_6MM_29Oct2025_suvr.csv"))
dftype = "suvr"
df.head(5)

,loniuid,ptid,rid,rh.lateralorbitofrontal,rh.parsorbitalis,rh.frontalpole,rh.medialorbitofrontal,rh.parstriangularis,rh.parsopercularis,rh.rostralmiddlefrontal,...,lh.transversetemporal,lh.insula,Left-Thalamus-Proper,Left-Caudate,Left-Putamen,Left-Pallidum,Left-Accumbens-area,Left-Hippocampus,Left-Amygdala,Brain-Stem
0,1602753,141_S_0767,767,1.207,1.128,0.978,1.029,1.098,1.037,1.008,...,1.077,1.102,1.086,0.943,1.359,1.734,0.832,1.126,0.959,1.630
1,1598063,037_S_4214,4214,1.082,1.010,0.772,0.872,1.036,0.972,0.974,...,0.979,1.008,1.277,1.002,1.261,1.699,0.955,1.016,1.001,1.688
2,1598059,037_S_4214,4214,1.101,1.003,0.733,0.956,1.011,0.983,0.935,...,1.010,1.001,1.178,0.948,1.249,1.532,0.952,0.981,0.999,1.568
3,1614231,037_S_4214,4214,1.097,1.002,0.815,0.911,1.059,1.017,1.005,...,0.987,1.010,1.241,0.987,1.326,1.727,0.991,0.953,1.056,1.672
4,1594117,006_S_4485,4485,1.723,1.737,1.666,1.661,1.757,1.765,1.741,...,0.943,1.218,1.324,1.042,1.359,1.707,1.127,1.141,1.004,1.709


####  Mapping of ADNI/oxMBM to preffered names

In [30]:
dnnames_mapping = pd.read_csv(os.path.join(base_PATH, "resources/adni_pet_image_analysis/structured_files_UCBERKELEY_AMY_6MM_29Oct2025/UCBERKELEY_AMY_6MM_29Oct2025_mapping.csv"))

map_dict = dict(zip(dnnames_mapping['dnnames'], dnnames_mapping['fsname']))

In [31]:
info_df = pd.merge(node_df, dnnames_mapping, how = 'left', left_on = 'dn_name', right_on = 'dnnames')
assert info_df.shape[0] == node_df.shape[0], "Not a One to One to merger"

In [32]:
# rename the columns of the data file
df.rename(columns=map_dict, inplace=True)

# rename graph
for node in graph.nodes():
    graph.nodes[node]['region_name'] = map_dict[graph.nodes[node]['dn_name']]
### Create new graph from filtered node properties if needed

In [73]:
graph.nodes[1]

{'dn_position_x': 34.0419230769,
 'dn_position_y': 82.1757692308,
 'dn_position_z': 31.7769230769,
 'dn_correspondence_id': 1,
 'dn_region': 'cortical',
 'dn_fsname': 'lateralorbitofrontal',
 'dn_name': 'rh.lateralorbitofrontal',
 'dn_hemisphere': 'right',
 'region_name': 'rh_lateralorbitofrontal',
 'suvr': np.float64(1.101)}

#### Setting {var_type} data into nx.Graph

In [58]:
print(f"Imported DF contains data for: {dftype} data of patients")
selected_patient_RID = 4214
df_id = np.random.choice(df[df['rid'] == selected_patient_RID].index)

Imported DF contains data for: suvr data of patients


In [59]:
for node in graph.nodes():
    graph.nodes[node][dftype] = df.loc[df_id, graph.nodes[node]['region_name']]

#### Export graph as graphml (for Gephi)

In [56]:
from utilsA1 import export_graphml_with_namespace

export_graphml_with_namespace(
    graph,
    output_path=os.path.realpath(os.path.join(base_PATH, "resources/suvr_updated.graphml")),
    xmlns_path="file:///C:/Users/sranasin/PycharmProjects/NetworkModels/resources/graphml.xsd"
)

np.float64(0.733)

### 3D-Visualization

In [75]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"


In [76]:
# Extract 3D positions from node attributes
pos = {}
for node, data in graph.nodes(data=True):
    x = float(data.get('dn_position_x', 0))
    y = float(data.get('dn_position_y', 0))
    z = float(data.get('dn_position_z', 0))
    pos[node] = (x, y, z)


In [77]:
edge_x, edge_y, edge_z = [], [], []
for u, v, edge_data in graph.edges(data=True):
    x0, y0, z0 = pos[u]
    x1, y1, z1 = pos[v]
    edge_x += [x0, x1, None] # Edge coordinates
    edge_y += [y0, y1, None]
    edge_z += [z0, z1, None]

In [78]:
# NOde coordinates
x_nodes = [pos[n][0] for n in graph.nodes()]
y_nodes = [pos[n][1] for n in graph.nodes()]
z_nodes = [pos[n][2] for n in graph.nodes()]

# Name and color each Node with regions and SUVR
labels = [graph.nodes[n].get('region_name', str(n)) for n in graph.nodes()]
suvr_vals = [graph.nodes[node].get('suvr', 0) for node in graph.nodes()]
min_suvr = min(suvr_vals)
max_suvr = max(suvr_vals)
node_color = [(suvr - min_suvr) / (max_suvr - min_suvr) for suvr in suvr_vals]

In [79]:

# --- 5. Plot edges and nodes in Plotly ---
edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='lightgray', width=1),
    hoverinfo='none'
)

node_trace = go.Scatter3d(
    x=x_nodes, y=y_nodes, z=z_nodes,
    mode='markers+text',
    text=labels,
    textposition="top center",
    marker=dict(
        size=20,
        color=node_color,
        colorscale='reds',
        colorbar=dict(title = "SUVR Uptake", tickvals=[0, 1]),
        line=dict(width=0.5, color='darkblue')
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "X: %{x:.2f}<br>"
        "Y: %{y:.2f}<br>"
        "Z: %{z:.2f}<extra></extra>"
    )
)

In [80]:
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    scene=dict(
        xaxis=dict(showbackground=False),
        yaxis=dict(showbackground=False),
        zaxis=dict(showbackground=False),
    ),
    margin=dict(l=0, r=0, b=0, t=30),
    title="3D Brain Connectivity Graph - DK33"
)

fig.show()

In [81]:
from utilsA1 import export_graphml_with_namespace

export_graphml_with_namespace(
    graph,
    output_path=os.path.realpath(os.path.join(base_PATH, "resources/suvr_updated.graphml")),
    xmlns_path="file:///C:/Users/sranasin/PycharmProjects/NetworkModels/resources/graphml.xsd"
)

GraphML exported to: C:\Users\sranasin\PycharmProjects\NetworkModels\resources\suvr_updated.graphml
